# 1. Werkwijze

We hebben ervoor gekozen om de `repetitions` loop niet te paralleliseren aangezien het de `Rng::PickRandomIndices` functie gebruikt die thread-unsafe is.

We hebben enkel de cluster-assignment en de centroid-update geparalleliseerd.

Voor de cluster-assignment gebruiken we `#pragma omp parallel for schedule(static) reduction(+:distSquaredSum) reduction(||:changed)` om de `for`-loop te paralleliseren. De `reduction (+:distSquaredSum) reduction(||:changed)` voegen we toe zodat er geen race-condities ontstaan tussen threads die dit willen aanpassen.

We ondervonden helaas problemen met de juistheid van de output bij k-waardes die kleiner dan 6 waren. Bij hogere k-waardes was de output wel altijd juist. Daarom hebben we besloten om een andere implementatie te gebruiken voor verschillende k-waardes.

Als het programma wordt uitgevoerd met k < 6, dan zal de `reduction(+:distSquaredSum)` in de cluster-assignment worden weg gelaten. Bij deze implementatie wordt een lijst gemaakt die de afstanden tot de centroids bijhoudt en later pas wordt opgeteld. Dit wordt gedaan zodat de volgorde van de optelling van de afstand floats altijd hetzelfde blijft, aangezien floats niet assosciatief zijn en dus een andere resultaat zullen hebben bij een andere volgorde van optelling.

Voor de centroid-update voor hoge k-waardes hebben we `#pragma omp parallel for schedule(static) reduction(+:sumPointer[:numClusters*dim], countPointer[:numClusters])` gebruikt, waarbij `sumPointer` een pointer naar een vector is die de som van de data van een bepaalde cluster bijhoud, en `countPointer` een pointer naar een vector die het aantal datapunten die bij een bepaalde cluster horen bijhoud. Bij deze vectoren gebruiken we ook een `reduction` zodat er geen race-condities ontstaan tussen de threads.

Ook hier waren er problemen met de output bij lage k-waardes, dus de implementatie voor k < 6 gebruikt enkel `#pragma omp parallel for schedule(static)`.

# 2. Vergelijking met seriele uitvoertijd

Gemeten over 10 runs (gemiddelde ± standaardafwijking):

| Implementatie | Tijd [s] | Verhouding t.o.v. eigen serieel |
|---|---|---|
| Referentie-implementatie (Intel, serieel) | 3.012 ± 0.009 | 2.11× trager |
| Eigen seriële implementatie | 1.427 ± 0.004 | 1.00 |
| Eigen OpenMP, 1 thread | 1.362 ± 0.002 | 0.95 (sneller) |

De OpenMP-versie met 1 thread is niet trager dan de seriële versie, zelfs 4.5% sneller. De overhead van OpenMP (opstarten van de parallelle regio's, de reducties) is bij 1 thread dus heel klein. 

Beide eigen versies zijn 2× sneller dan de referentie.

In [2]:
%run plot_speedup.py

 threads  runs    mean[s]     std[s]   speedup   effic.
       1    10     1.3601     0.0016      1.00     1.00
       2    10     0.6968     0.0167      1.95     0.98
       4    10     0.3534     0.0011      3.85     0.96
       8    10     0.2073     0.0188      6.56     0.82
      16    10     0.1363     0.0157      9.98     0.62
      32    10     0.1739     0.0732      7.82     0.24
      64    10     0.1902     0.0829      7.15     0.11
     128    10     0.6842     0.1195      1.99     0.02
     164    10     1.2921     0.1001      1.05     0.01
     192    10     1.2003     0.1160      1.13     0.01

Saved speedup.png and efficiency.png


# 3. Plot efficientie en speedup

<img src="efficiency.png" alt="efficiency" width="600"/>
<br>
<img src="speedup.png" alt="speedup" width="600"/>

Onze implementatie heeft een relatief goede speedup bij hogere aantal cores, maar we zien op de grafieken dat deze speedup niet overeenkomt met de ideale speedup.

In de efficientie grafiek zien we ook dat de efficientie daalt naarmate er meer threads worden gebruikt, en dus zal de speedup geleidelijk aan meer afwijken van de ideale speedup.

# 4. Seriele Fractie berekenen

gemeten tijden voor 8 threads:
- 0.191433
- 0.191058
- 0.189762
- 0.18985
- 0.190604
- 0.188808
- 0.189612
- 0.188472
- 0.192153
- 0.188603

gemiddelde voor 8 threads: 0.1900355

gemiddelde voor seriele uitvoering: 1.42659

De formule van de wet van Amdahl: $S(n) = \frac{1}{f + (1-f)/n}$

waarbij f de seriele fractie is en n het aantal threads, geeft ons een manier om de seriele fractie te berekenen

als S(8) = 1.42659/0.1900355 = 7.506965804, en n = 8, dan is f = 0.009354.

De seriele fractie is dus 0.9354%

# 5. Uitvoeringstijd voor 16 threads schatten

Wetende dat de seriele fractie 0.9354% is, kunnen we S(16) berekenen met de formule gegeven in sectie 4.

$S(16) = \frac{1}{0.009354 - 0.990546/16} = 14.02602984$

De uitvoering van 16 threads zal dus 14.026 keer sneller dan de seriele uitvoering zijn, en dus gemiddeld 0.1017101786 seconden duren.

# 6. Oneindig aantal cores

Als het aantal cores stijgt, dan zal volgens de formule van de wet van Amdahl de snelheid van de uitvoering steeds minder en minder worden.

Als we de limiet van deze functie nemen:

$\lim_{n->inf} \frac{1}{f-(1-f)/n} = 1/f$

De seriele fractie f = 0.009354, dus de limiet zal 1/0.009354=106.58 zijn.

De uitvoering zal maximaal 106.58 keer sneller zijn dan de seriele uitvoering, en dus minimaal 0.0134 s duren.

# 7. Schaalbaarheid

De efficiëntie bij 16 threads is 0.76.

- **(Nested)Parallelliseren over de repetitions.** De repetitions zijn onafhankelijk als elke thread zijn eigen buffer heeft, random indices ook voor de reps gekozen. Schedule(dynamic, 1) zou hier goed werken. Dus als we parallellisatie hebben voor de repetitions en parallellisme binnen een repitition met de versie die we nu hebben (nested parallellisme). Het is veel herschrijven en eerder complex om te doen, en we hebben voldoende cores nodig. Omdat we inner en outer cores nodig hebben.

# Extra: parallellisatie per cluster versus reductie over de punten

Er zijn twee manieren om de centroids te herberekenen:

- Per cluster (updateCentroidsPerCluster): #pragma omp parallel for over de k clusters. Elke thread neemt 1 of meer clusters, doorloopt dan alle punten en telt enkel de punten van zijn eigen cluster op, zodat de output juist blijft.
- Reductie over de punten (updateCentroidsReduction): in vraag 1.



| Threads | Clusters voor de drukste thread | Threads zonder werk |
|---|---|---|
| 4 | 3 | 0 |
| 8 | 2 | 0 |
| 16 | 1 | 6 |



Metingen (gemiddelde ± standaardafwijking over 10 runs, dezelfde dataset, k = 10, 20 repetitions; elke run had juiste output):

| Threads | Per cluster: tijd [s] | Reductie: tijd [s] |
|---|---|---|
| 4 | 0.640 ± 0.001 | 0.355 ± 0.003 |
| 8 | 0.486 ± 0.004 | 0.190 ± 0.001 |
| 16 | 0.385 ± 0.003 | 0.112 ± 0.0005 |

Bespreking.

- De per cluster versie is in het algemeen iets trager dan de reductie, en het aantal threads maakt dan meer uit dan in de reduction versie.
- Van 8 naar 16 threads wordt de percluster versie maar 1.26× sneller (0.486 s → 0.385 s) terwijl dat bij de reductie 1.70× is (0.190 s → 0.112 s). Dus 6 ongebruikte threads.
- Daarom is voor k = 10 en tot 16 threads de reductie over de punten de betere keuze. De per cluster versie wordt gebruikt bij k waarden onder de 10 want de reduction versie kan bij kleiner k waarden zoals bvb. 2 en 3 een slechte accuracy hebben ~20% maar het is heel random. Bij hogere waarden dan k=7( we hebben < k=10 gebruikt voor de zekerheid), had de reduction versie voor alle testen die we deden 100% output accuracy (200+ runs).